# RunOptions

`pyneat.RunOptions` configures how a built Neat runtime behaves. It does not change the graph topology or model math. It controls runtime policy: queues, overflow behavior, output tensor ownership, startup checks, optional export, optional power monitoring, and a few advanced safety knobs.

You pass it when you build or run a graph:

```python
run = graph.build(run_options)
run = graph.build([seed_tensor], run_options)
outputs = graph.run([tensor], run_options)
```

This notebook uses a simple RGB pass-through graph so the runtime behavior is easy to see without needing a model archive.

## Imports

Run this notebook from the DevKit pyneat environment.

In [1]:
from pathlib import Path
import numpy as np
import pyneat


## Mental Model

`Graph` describes what should be connected. `RunOptions` describes how the runtime should behave after the graph is built.

```text
Graph nodes and groups
  -> graph.build(run_options)
  -> Run handle
  -> push / pull / run / measure / close
```

For batch or correctness-first work, prefer blocking and owned outputs. For live video, prefer small queues and a policy that keeps fresh frames.

## Complete RunOptions Field Map

| Field | Default | What it controls |
| --- | --- | --- |
| `preset` | `RunPreset.Balanced` | High-level latency/safety profile used by the runtime when it resolves defaults. |
| `queue_depth` | `4` | Capacity of runtime input/output queues. Larger queues absorb jitter; smaller queues protect latency. |
| `overflow_policy` | `OverflowPolicy.Block` | What happens when an input queue is full. |
| `output_memory` | `OutputMemory.Auto` | Whether pulled tensors are owned copies or zero-copy views into runtime buffers. |
| `input_timeout_ms` | `-1` | Default input-side timeout; `-1` keeps framework defaults. Explicit `pull(..., timeout_ms)` still overrides per call. |
| `startup_preflight` | `True` | For seeded builds, push/pull the seed once during build to catch payload-level failures early. |
| `advanced` | `RunAdvancedOptions()` | Expert runtime safety and memory knobs. |
| `power_monitor` | disabled | Optional board rail power telemetry configuration. |
| `run_export` | disabled | Optional build-time run/graph JSON export. |
| `graph_run_export` | disabled | Backward-compatible export field name. |
| `on_input_drop` | unset | Optional callback invoked when the runtime drops an input frame. |

In [2]:
def public_names(obj):
    return [name for name in dir(obj) if not name.startswith("_")]

default_run_options = pyneat.RunOptions()
print("RunOptions fields:")
print(public_names(default_run_options))
print()
print("Defaults:")
print("preset:", default_run_options.preset)
print("queue_depth:", default_run_options.queue_depth)
print("overflow_policy:", default_run_options.overflow_policy)
print("output_memory:", default_run_options.output_memory)
print("input_timeout_ms:", default_run_options.input_timeout_ms)
print("startup_preflight:", default_run_options.startup_preflight)


RunOptions fields:
['advanced', 'disable_power_monitor', 'enable_board_power', 'enable_modalix_dvt_power', 'enable_modalix_som_power', 'graph_run_export', 'input_timeout_ms', 'on_input_drop', 'output_memory', 'overflow_policy', 'power_monitor', 'preset', 'queue_depth', 'run_export', 'startup_preflight']

Defaults:
preset: RunPreset.Balanced
queue_depth: 4
overflow_policy: OverflowPolicy.Block
output_memory: OutputMemory.Auto
input_timeout_ms: -1
startup_preflight: True


## Runtime Enums

### `RunPreset`

| Value | Use when |
| --- | --- |
| `Realtime` | You care most about low latency and fresh frames. |
| `Balanced` | You want the default general-purpose runtime profile. |
| `Reliable` | You prefer conservative, lossless behavior and stable output ownership. |

### `OverflowPolicy`

| Value | Queue-full behavior | Typical use |
| --- | --- | --- |
| `Block` | Wait until queue space is available. | Batch processing, correctness checks, file inputs. |
| `KeepLatest` | Drop older queued work to make room for the newest input. | Live camera and RTSP analytics. |
| `DropIncoming` | Reject the new input and keep queued work. | Pipelines where queued work should finish in order. |

### `OutputMemory`

| Value | Meaning | Typical use |
| --- | --- | --- |
| `Auto` | Let Neat choose based on preset, mode, and platform. | Default starting point. |
| `ZeroCopy` | Pulled tensors may share runtime buffer storage. | Fast graph-to-graph or immediate-use paths. |
| `Owned` | Pulled tensors own/copy their data. | Notebooks, CPU inspection, storing outputs, debug code. |

In [3]:
print("RunPreset:", public_names(pyneat.RunPreset))
print("OverflowPolicy:", public_names(pyneat.OverflowPolicy))
print("OutputMemory:", public_names(pyneat.OutputMemory))


RunPreset: ['Balanced', 'Realtime', 'Reliable']
OverflowPolicy: ['Block', 'DropIncoming', 'KeepLatest']
OutputMemory: ['Auto', 'Owned', 'ZeroCopy']


## Presets And Explicit Fields

A preset is a profile. It influences resolved runtime defaults such as queue depth, overflow behavior, internal polling, and output memory policy. Explicit field assignments still make your intent clear and are the preferred tutorial style.

| Workload | Good starting point |
| --- | --- |
| Live video / RTSP | `Realtime`, small `queue_depth`, `KeepLatest` |
| Batch or image folder | `Reliable` or `Balanced`, `Block`, `Owned` |
| Notebook/debug inspection | `Balanced`, `Block`, `Owned` |
| High-throughput graph forwarding | `Realtime` or `Balanced`, measured `queue_depth`, often `ZeroCopy` |

In [4]:
def notebook_debug_options():
    opt = pyneat.RunOptions()
    opt.preset = pyneat.RunPreset.Balanced
    opt.queue_depth = 4
    opt.overflow_policy = pyneat.OverflowPolicy.Block
    opt.output_memory = pyneat.OutputMemory.Owned
    return opt


def live_video_options(output_memory=pyneat.OutputMemory.Owned):
    opt = pyneat.RunOptions()
    opt.preset = pyneat.RunPreset.Realtime
    opt.queue_depth = 3
    opt.overflow_policy = pyneat.OverflowPolicy.KeepLatest
    opt.output_memory = output_memory
    return opt


def reliable_batch_options():
    opt = pyneat.RunOptions()
    opt.preset = pyneat.RunPreset.Reliable
    opt.queue_depth = 8
    opt.overflow_policy = pyneat.OverflowPolicy.Block
    opt.output_memory = pyneat.OutputMemory.Owned
    return opt


for name, opt in [
    ("notebook", notebook_debug_options()),
    ("live", live_video_options()),
    ("batch", reliable_batch_options()),
]:
    print(name, opt.preset, opt.queue_depth, opt.overflow_policy, opt.output_memory)


notebook RunPreset.Balanced 4 OverflowPolicy.Block OutputMemory.Owned
live RunPreset.Realtime 3 OverflowPolicy.KeepLatest OutputMemory.Owned
batch RunPreset.Reliable 8 OverflowPolicy.Block OutputMemory.Owned


## Build A Tiny Graph

This pass-through graph accepts an RGB image tensor and returns an RGB image tensor. It is intentionally boring: that keeps the focus on runtime options.

In [5]:
HEIGHT = 120
WIDTH = 160

image_rgb = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
image_rgb[..., 0] = 32
image_rgb[..., 1] = 96
image_rgb[..., 2] = 160
tensor = pyneat.Tensor.from_numpy(image_rgb, copy=True, image_format=pyneat.PixelFormat.RGB)

input_options = pyneat.InputOptions()
input_options.format = pyneat.Format.RGB
input_options.width = WIDTH
input_options.height = HEIGHT
input_options.depth = 3
input_options.is_live = True

graph = pyneat.Graph("run_options_pass_through")
graph.add(pyneat.nodes.input(input_options))
graph.add(pyneat.nodes.output("out"))

print(graph.describe())


0) Input  [mysrc]
1) Output  [out]


## Build With RunOptions

`RunOptions` is applied when the graph becomes a `Run`. A seeded build uses the seed tensor to establish the input contract and, when `startup_preflight` is enabled, validates the path once during build.

In [6]:
run_options = notebook_debug_options()

run = graph.build([tensor], run_options)
try:
    outputs = run.run([tensor], timeout_ms=1000)
    out = outputs[0].to_numpy(copy=True)
    print("output shape:", out.shape)
    print("same pixels:", bool(np.array_equal(out, image_rgb)))
finally:
    run.close()


output shape: (120, 160, 3)
same pixels: True


[1/4] Initializing runtime...
[2/4] Preparing input stream...
[3/4] Building graph...
[4/4] Graph ready (5 ms)


## `queue_depth`

`queue_depth` is the runtime buffer capacity. Bigger values can absorb bursty producers or variable downstream latency. Smaller values reduce how much stale work can accumulate.

Practical starting points:

| Scenario | Starting `queue_depth` |
| --- | --- |
| Notebook or simple debug run | `4` |
| Live RTSP/camera analytics | `2` to `4` |
| Production model path with jitter | `4` to `16`, then measure |
| Lossless batch workload | `8` or more if memory allows |

## `overflow_policy`

`overflow_policy` decides what happens when the producer is faster than the runtime can consume.

| Policy | Effect | Latency behavior |
| --- | --- | --- |
| `Block` | Producer waits. No intentional frame loss. | Latency can grow if upstream is live. |
| `KeepLatest` | Old queued input is replaced by newer input. | Good for live freshness. |
| `DropIncoming` | New input is rejected while queued input remains. | Preserves queued order but drops new frames. |

For live video, stale frames are usually worse than dropped frames, so `KeepLatest` is the common first choice.

## Configure Queue And Overflow Policy

These settings are usually chosen together. A large queue with `KeepLatest` still allows more stale work to accumulate than a small queue. A small queue with `Block` protects memory but may stall the producer. Start with a workload recipe, then measure.

In [7]:
def summarize_policy(name, opt):
    print(
        name,
        "preset=", opt.preset,
        "queue_depth=", opt.queue_depth,
        "overflow_policy=", opt.overflow_policy,
    )


low_latency_live = live_video_options()

lossless_batch = reliable_batch_options()

ordered_backlog = pyneat.RunOptions()
ordered_backlog.preset = pyneat.RunPreset.Balanced
ordered_backlog.queue_depth = 8
ordered_backlog.overflow_policy = pyneat.OverflowPolicy.DropIncoming
ordered_backlog.output_memory = pyneat.OutputMemory.Owned

summarize_policy("low_latency_live", low_latency_live)
summarize_policy("lossless_batch", lossless_batch)
summarize_policy("ordered_backlog", ordered_backlog)


low_latency_live preset= RunPreset.Realtime queue_depth= 3 overflow_policy= OverflowPolicy.KeepLatest
lossless_batch preset= RunPreset.Reliable queue_depth= 8 overflow_policy= OverflowPolicy.Block
ordered_backlog preset= RunPreset.Balanced queue_depth= 8 overflow_policy= OverflowPolicy.DropIncoming


## Drop Callback

`RunOptions.on_input_drop` lets an application observe frames that the runtime drops because an overflow policy rejected or replaced input. The callback receives a `pyneat.InputDropInfo` object with metadata such as `frame_id`, `stream_id`, `format`, dimensions, input port, and `reason`.

The callback should be lightweight because it runs on the pushing thread. Use it for counters, logging, or telemetry; do not do slow processing inside it.

The next cell only installs the callback. Drop events appear only after a run is built with these options and the input queue actually overflows.


In [8]:
drop_events = []

def record_drop(info):
    drop_events.append({
        "frame_id": info.frame_id,
        "stream_id": info.stream_id,
        "reason": info.reason,
    })

drop_options = live_video_options()
drop_options.on_input_drop = record_drop

print("callback installed:", drop_options.on_input_drop is not None)


callback installed: True


## `output_memory`

`output_memory` is about output tensor lifetime.

- Use `Owned` when Python will inspect, store, display, or pass the tensor outside the immediate pull step.
- Use `ZeroCopy` when data stays inside a tightly controlled runtime path and you have measured that copies matter.
- Use `Auto` when you want Neat to select the policy from the preset and run mode.

For notebooks, `Owned` is usually the least surprising setting.

In [9]:
owned_for_python = notebook_debug_options()
owned_for_python.output_memory = pyneat.OutputMemory.Owned

zero_copy_for_forwarding = live_video_options(pyneat.OutputMemory.ZeroCopy)

print("owned_for_python:", owned_for_python.output_memory)
print("zero_copy_for_forwarding:", zero_copy_for_forwarding.output_memory)


owned_for_python: OutputMemory.Owned
zero_copy_for_forwarding: OutputMemory.ZeroCopy


## Timeouts And Startup Preflight

`input_timeout_ms` sets the default timeout for input-mode build/run paths. Leave it at `-1` unless you need a process-wide default; explicit pull timeouts are clearer in notebooks.

`startup_preflight` applies to seeded builds such as `graph.build([tensor], options)`. When true, Neat validates the seed path during build so invalid payloads fail early. Turn it off only when startup latency matters and you are prepared to handle first-sample errors later.

In [10]:
fast_start_options = live_video_options()
fast_start_options.input_timeout_ms = 2000
fast_start_options.startup_preflight = False

print("input_timeout_ms:", fast_start_options.input_timeout_ms)
print("startup_preflight:", fast_start_options.startup_preflight)


input_timeout_ms: 2000
startup_preflight: False


## Advanced Options

`RunOptions.advanced` is for expert tuning and safety limits. Most notebooks and applications should leave these at their defaults.

| Field | Default | Use when |
| --- | --- | --- |
| `advanced.copy_input` | `False` | The producer's input buffer may be short-lived and you want defensive copies. |
| `advanced.max_input_bytes` | `0` | You want to reject unexpectedly large inputs; `0` means no cap. |
| `advanced.sync_num_buffers_override` | `-1` | You need to override appsrc buffer count for sync-mode runs. |
| `advanced.prepare_output_cpu_visible` | `False` | Zero-copy outputs need CPU visibility preparation before entering the output queue. |

In [11]:
advanced_options = pyneat.RunOptions()
print("Advanced fields:", public_names(advanced_options.advanced))
print("copy_input:", advanced_options.advanced.copy_input)
print("max_input_bytes:", advanced_options.advanced.max_input_bytes)
print("sync_num_buffers_override:", advanced_options.advanced.sync_num_buffers_override)
print("prepare_output_cpu_visible:", advanced_options.advanced.prepare_output_cpu_visible)


Advanced fields: ['copy_input', 'max_input_bytes', 'prepare_output_cpu_visible', 'sync_num_buffers_override']
copy_input: False
max_input_bytes: 0
sync_num_buffers_override: -1
prepare_output_cpu_visible: False


## Build-Time Run Export

`run_export` can write a JSON snapshot when the run is built. This is useful for CI artifacts and graph inspection. Runtime measurements should still be captured after work has executed.

Common fields:

| Field | Meaning |
| --- | --- |
| `run_export.path` | Output JSON path; empty disables export. |
| `run_export.label` | Optional label in the export. |
| `run_export.include_metrics` | Include available metrics. |
| `run_export.include_power` | Include available power data. |
| `run_export.include_node_metrics` | Include node-level latency rows. |
| `run_export.include_plugin_metrics` | Include plugin/kernel rows when available. |
| `run_export.indent` | JSON indentation. |

In [12]:
export_options = notebook_debug_options()
export_options.run_export.path = "/tmp/neat_run_options_demo.json"
export_options.run_export.label = "run-options-demo"
export_options.run_export.include_plugin_metrics = False
export_options.run_export.indent = 2

print("export path:", export_options.run_export.path)
print("export label:", export_options.run_export.label)


export path: /tmp/neat_run_options_demo.json
export label: run-options-demo


## Power Monitoring

Power monitoring is optional. Enable it when you want latency, throughput, and board rail telemetry from the same measured run.

```python
run_options = pyneat.RunOptions()
run_options.enable_board_power(sample_interval_ms=100)
run = graph.build([tensor], run_options)
scope = run.start_measurement()
# push/pull workload
report = scope.stop()
```

Use this on hardware where the board power profile is available.

In [13]:
power_options = pyneat.RunOptions()
power_options.enable_board_power(100)
print("power enabled:", power_options.power_monitor.enabled)
print("sample interval ms:", power_options.power_monitor.sample_interval_ms)
power_options.disable_power_monitor()
print("power enabled after disable:", power_options.power_monitor.enabled)


power enabled: True
sample interval ms: 100
power enabled after disable: False


## RunOptions vs OutputOptions

These option objects control different boundaries.

| Object | Applied at | Controls |
| --- | --- | --- |
| `pyneat.RunOptions` | `graph.build(...)` or `graph.run(...)` | Runtime queueing, overflow policy, output memory, startup checks, export, power. |
| `pyneat.OutputOptions` | `pyneat.nodes.output(...)` | Buffering, dropping, clock sync, and combine policy at one graph output endpoint. |

A live stream often uses both: small output endpoint buffering plus `RunOptions` that keep the freshest frame.

## Decision Guide

| Goal | Settings to start with |
| --- | --- |
| Safest notebook/debug output | `Balanced`, `queue_depth = 4`, `Block`, `Owned` |
| Lossless batch processing | `Reliable`, deeper queue, `Block`, `Owned` |
| Fresh live camera frames | `Realtime`, `queue_depth = 2` to `4`, `KeepLatest` |
| Avoid building a stale live backlog | Small queue plus `KeepLatest` |
| Preserve queued order under overload | `DropIncoming` |
| Minimize output copies | `ZeroCopy`, then measure and keep tensor lifetime local |
| Keep output tensors after pull | `Owned` |

## Common Mistakes

- Using `Block` for a live camera and then wondering why latency grows.
- Using `ZeroCopy` in notebook/debug code, then keeping tensors longer than the runtime buffer lifetime.
- Increasing `queue_depth` to improve throughput without measuring latency.
- Setting `RunOptions` after `graph.build(...)`; options are applied when the run is built.
- Treating `RunOptions` as graph structure. It is runtime behavior, not graph composition.
- Ignoring `pull()` timeout and closed/error cases in long-running loops.

## What To Remember

- `RunOptions` configures a built runtime, not the graph's math.
- `preset`, `queue_depth`, `overflow_policy`, and `output_memory` are the everyday fields.
- Live streams usually want `Realtime`, small queues, and `KeepLatest`.
- Batch and debugging usually want `Block` and `Owned`.
- Use `Owned` when Python will inspect or keep output data.
- Use `ZeroCopy` only when you understand tensor lifetime and have measured the copy cost.
- Advanced, export, and power options are useful, but they should follow a clear measurement or diagnostics goal.

## References

- Core API header: [Run.h](https://github.com/sima-neat/core/blob/main/include/pipeline/Run.h)
- Python bindings: [module.cpp](https://github.com/sima-neat/core/blob/main/python/src/module.cpp)
- Core queue tutorial: [tune_throughput_and_queues.py](https://github.com/sima-neat/core/blob/main/tutorials/016_tune_throughput_and_queues/tune_throughput_and_queues.py)
- Core graph execution tutorial: [build_inference_pipeline.py](https://github.com/sima-neat/core/blob/main/tutorials/004_build_inference_pipeline/build_inference_pipeline.py)
- Official docs: [Develop Apps with SiMa.ai Neat](https://developer.sima.ai/software/develop-apps/) and [Tutorials](https://developer.sima.ai/software/tutorials/)